In [1]:
!pip install pandas numpy matplotlib

In [ ]:
import os, re, sys, json, csv, gzip, string
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
file_path = "/content/drive/MyDrive/CAISI_phase2/safe-finetuning/SaLoRA/outputs/debug_run/checkpoint-1330/trainer_state.json"

In [ ]:
trainer_state_data = []
with open(file_path, 'r') as file:
  data = json.load(file) #Open JSON file and loads it into a dictionary

#looks like:
#{
#  "best_model_checkpoint": "checkpoint-1330",
# "log_history": [
#   {"loss": 1.234, "learning_rate": 1e-5, "step": 10},
#   {"loss": 1.123, "learning_rate": 1e-5, "step": 20},
#   {"eval_loss": 1.045, "eval_accuracy": 0.82, "step": 30}
# ],
# "epoch": 3.0,
#  ...
#}

df = pd.json_normalize(data, record_path=['log_history']) #Pandas function that flattens dictionary into a dataframe;
#makes each dictionary in the list at the key log_history a row in the dataframe

In [ ]:
df

In [ ]:
  log_history = pd.DataFrame(data.get("log_history", []))
  log_history = log_history[(log_history["loss"].notna()) | (log_history["eval_loss"].notna())]

  df_melted = log_history.melt(
      id_vars=["step"],
      value_vars=["loss", "eval_loss"],
      var_name="Loss Type",
      value_name="Loss"
  )

#Plot loss vs. steps

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10,6))
sns.lineplot(
    data=df_melted,
    x="step", y="Loss", hue="Loss Type",
    style="Loss Type", markers=True, dashes=False
)
plt.xlabel("Training Steps", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("Training vs. Validation Loss", fontsize=16)
plt.legend(title="Loss Type")
plt.show()